# DSA и ECDSA

Мы изучили, как проблема дискретного логарифма может быть использована для шифрования. DH и ECDH позволяют создать симметричный ключ при помощи асимметричной криптографии. К сожалению, их нельзя использовать для создания подписи. Но группы по модулю простого числа и группы точек эллиптических кривых можно использовать для создания подписи. Самые распространённые алгоритмы для этого - это DSA и его эллиптический вариант ECDSA.

## Принцип работы 

DSA нужна хеш-функция $\mathit{Hash}(x)$, мультипликативная группа $Z^*_p$ по модулю простого числа $p$ и генератор  $g$ порядка $q$, где $q$ - наибольший простой делитель $p-1$ (т.е. $g$ - генератор наибольшей подгруппы простого порядка группы $Z^*_p$). Мы уже это проходили, но чтобы найти такой генератор, надо выбрать случайный элемент $z \ne 1,p-1$ и вычислить $g=z^\frac{p-1}{q}\  mod\  p$. Очевидно, все эти параметры обычно стандартизированы.

Теперь, когда у нас есть $\mathit{Hash}(x)$, $Z^*_p$, $g$, $q$, мы можем создать ключевую пару. Ключи генерируются следующим образом:

1. Выбираем случайный $x\in(0,q)$ - это закрытый ключ

2. Вычисляем $y=g^x\  mod\  p$ - это открытый ключ

Теперь можно подписать сообщение $m$:

1. Выбираем случайный nonce (number used once, число, используемое один раз) $k\in (0,q)$

2. Вычисляем  $r = (g^k\  mod\  p)\  mod\  q$. Если $r=0$ идём на 1.

3. Вычисляем $s = \frac{\mathit{Hash}(m)+x\cdot r}{k}\  mod\  q$. Если $s=0$ идём на 1.

4. Выводим $(r,s)$. Это подпись.

Для проверки подписи мы должны подтвердить, что $r,s \in (0,q)$ и проверить следующее равенство:
$$(g^\frac{\mathit{Hash}(m)}{s}\cdot y^\frac{r}{s}\  mod\ p)\  mod \  q=r$$

Давайте проверим корректность:
$$(g^\frac{\mathit{Hash}(m)}{s}\cdot y^\frac{r}{s}\  mod\ p)\  mod \  q=(g^\frac{\mathit{Hash}(m)}{s}\cdot g^{x\cdot\frac{r}{s}}\  mod\ p)\  mod \  q$$

$$(g^\frac{\mathit{Hash}(m)}{s}\cdot g^{x\cdot\frac{r}{s}}\  mod\ p)\  mod \  q=(g^{\frac{\mathit{Hash}(m)}{s}+x\cdot\frac{r}{s}}\  mod\ p)\  mod \  q$$

$$(g^{\frac{\mathit{Hash}(m)}{s}+x\cdot\frac{r}{s}}\  mod\ p)\  mod \  q=(g^{\frac{\mathit{Hash}(m)+x\cdot r}{s}}\  mod\ p)\  mod \  q$$

$$(g^{\frac{\mathit{Hash}(m)+x\cdot r}{s}}\  mod\ p)\  mod \  q=(g^{\frac{\mathit{Hash}(m)+x\cdot r}{\left(\frac{\mathit{Hash}(m)+x\cdot r}{k}\right)}}\  mod\ p)\  mod \  q=(g^k\  mod\  p)\  mod\  q=r$$


В случае ECDSA у нас есть кривая $E(\mathit{GF}(p))$ и некоторая точка $G$ в качестве генератора, порядок которой - простое число $q$, так что $q\cdot G=0$ (Точка на Бесконечности). Нам также нужна хеш-функция $\mathit{Hash}(x)$. Для создания ключа мы снова:

1. Выбираем случайное $d \in (0,q)$ в качестве закрытого ключа

2. Вычисляем $Q=d\cdot G$, это наш открытый ключ

Алгоритм подписи сообщения $m$:

1. Выбираем случайный nonce $k\in(0,q)$

2. Вычисляем $(x_1,y_1)=k\cdot G$

3. Вычисляем $r=x_1\  mod \  q$. Если $r=0$, то возвращаемся на 1.

4. Вычисляем  $s=\frac{\mathit{Hash}(m)+r\cdot d}{k}\  mod\  q$. Если $s=0$, то возвращаемся на 1.

5. Выводим $(r,s)$ в качестве подписи.

Проверка:

1. Проверяем, что $r,s \in (0,q)$

2. Вычисляем $(x_1,y_1)=\frac{Hash(m)}{s}\cdot G+\frac{r}{s}\cdot Q$

3. Проверяем $r=x_1 \  mod \  q$

Корректность:
$$\frac{\mathit{Hash}(m)}{s}\cdot G+\frac{r}{s}\cdot Q=\frac{\mathit{Hash}(m)}{s}\cdot G+d\frac{r}{s}\cdot G=\frac{\mathit{Hash}(m)+d\cdot r}{s}\cdot G=k\cdot G=(x_1,y_1)$$

## Проблемы

Вы могли заметить, что оба примитива используют случайное значение $k$ при подписи. Это основная проблема этого семейства алгоритмов, потому что **очень многое** может пойти не так. Имплементации DSA и ECDSA могут быть взломаны, если:

1. Используется предсказуемое $k$

2. Одинаковое $k$ используется дважды

3. Используются $k$ со статистическим отклонением (но это мы здесь не рассмотрим)

В первом случае, раз мы можем предсказать $k$, мы можем восстановить закрытый ключ:

1. Для DSA. $s = \frac{\mathit{Hash}(m)+x\cdot r}{k}\  mod\  q \implies x=\frac{s\cdot k-\mathit{Hash}(m)}{r} \  mod \  q$

2. Для ECDSA $s=\frac{\mathit{Hash}(m)+r\cdot d}{k}\  mod\  q \implies d=\frac{s\cdot k-\mathit{Hash}(m)}{r} \  mod \  q$

Во втором сценарии, поскольку одинаковые $k$ были использованы для создания двух разных подписей:

$$k=\frac{\mathit{Hash}(m)+x\cdot r}{s}\  mod\  q \implies \frac{\mathit{Hash}(m_1)+x\cdot r_1}{s_1}=\frac{\mathit{Hash}(m_2)+x\cdot r_2}{s_2} \  mod\  q \implies$$

$$(\mathit{Hash}(m_1)+x\cdot r_1)s_2=(\mathit{Hash}(m_2)+x\cdot r_2)s_1 \  mod \  q \implies$$

$$\mathit{Hash}(m_1)s_2+x\cdot r_1\cdot s_2=\mathit{Hash}(m_2)s_1+x\cdot r_2\cdot s_1 \  mod \  q \implies$$

$$\mathit{Hash}(m_1)s_2-\mathit{Hash}(m_2)s_1=x(r_2\cdot s_1-r_1\cdot s_2)\  mod\  q \implies$$

$$x=\frac{\mathit{Hash}(m_1)s_2-\mathit{Hash}(m_2)s_1}{r_2\cdot s_1-r_1\cdot s_2}\  mod \  q$$

Т.е. мы снова можем вычислить $x$:

$$x=\frac{\mathit{Hash}(m_1)s_2-\mathit{Hash}(m_2)s_1}{r_2\cdot s_1-r_1\cdot s_2}\  mod \  q$$


## Задание
Поскольку атаки похожи, потренируемся только на DSA. Будет два подзадания. Сервер генерирует две ключевых пары. Он подписывает первое сообщение первой парой с использованием предсказуемого $k$:
```python
class predictable_DSA(DSA):
    def get_k(self):
        return int(time.time())%(self.q-1)+1
```
Подсказка: сервер может находиться в другой временной зоне.

Потом он генерирует новое  $k$ другой функцией и использует его для создания второй и третьей подписей. Для них используется второй ключ. Ваша цель создать подпись сообщения 'give' при помощи первого ключа и сообщения  'flag' при помощи второго ключа, а потом отправить их на сервер.
Удачи!

In [1]:
import random
import time
from Crypto.Util.number import bytes_to_long
import hashlib
#Классы, используемые сервером
class DSA:
    def __init__(self,g,p,q):
        self.p=p
        self.g=g
        self.q=q
        self.x=random.randint(1,q-1)
        self.y=pow(g,self.x,self.p)
    
    def get_k(self):
        raise NotImplementedError()
    @staticmethod
    def message_to_hashnum(m):
        if not isinstance(m,bytes):
            raise TypeError()
        return bytes_to_long(hashlib.sha256(m).digest())
    def sign(self,m):
        r=0
        s=0
        while r==0 or s==0:
            k=0
            while k==0:
                k=self.get_k()
            r=pow(g,k,self.p)%self.q
            s=(((DSA.message_to_hashnum(m)+(self.x*r)%self.q)%self.q)*pow(k,q-2,self.q))%self.q
        return (r,s)
    def verify(self,m,r,s,otherY):
        if r==0 or s==0: return False
        w=pow(s,self.q-2,self.q)
        u1=(DSA.message_to_hashnum(m)*w)%self.q
        u2=(r*w)%self.q
        res=(pow(self.g,u1,self.p)*pow(otherY,u2,self.p)%p)%q
        return res==r
    
    
class predictable_DSA(DSA):
    def get_k(self):
        return int(time.time())%(self.q-1)+1

class repeated_DSA(DSA):
    def get_k(self):
        global constant_k
        return constant_k


In [14]:
import socket
import re
class VulnServerClient:
    def __init__(self,show=True):
        """Инициализация, подключаемся к серверу"""
        self.s=socket.socket(socket.AF_INET,socket.SOCK_STREAM)
        self.s.connect(('cryptotraining.zone',1350))
       
    def recv_until(self,symb=b'\n>'):
        """Получаем сообщения от сервера, по умолчанию до приглашения"""
        data=b''
        while True:
            
            data+=self.s.recv(1)
            if data[-len(symb):]==symb:
                break
        return data
    def getChallenge(self,show=True):
        data=self.recv_until()
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Error decoding unicode. Try connecting to server again.')
            return (None,None)
        if show:
            print (data)
        p=int(re.search(r'(?<=p=)\d+',data).group(0))
        q=(p-1)//2
        g=int(re.search(r'(?<=g=)\d+',data).group(0))
        y1=int(re.search(r'(?<=y1=)\d+',data).group(0))
        y2=int(re.search(r'(?<=y2=)\d+',data).group(0))
        mrs1=re.search(r'(?<=s1\)=)\(b\'[a-z]+\',\d+,\d+\)',data).group(0)[1:-1].split(',')
        m1,r1,s1=mrs1[0][2:-1].encode(),int(mrs1[1]),int(mrs1[2])
        mrs2=re.search(r'(?<=s2\)=)\(b\'[a-z]+\',\d+,\d+\)',data).group(0)[1:-1].split(',')
        m2,r2,s2=mrs2[0][2:-1].encode(),int(mrs2[1]),int(mrs2[2])
        mrs3=re.search(r'(?<=s3\)=)\(b\'[a-z]+\',\d+,\d+\)',data).group(0)[1:-1].split(',')
        m3,r3,s3=mrs3[0][2:-1].encode(),int(mrs3[1]),int(mrs3[2])
        #print (m1.encode(),r1,s1)
        return (p,q,g,y1,y2,m1,r1,s1,m2,r2,s2,m3,r3,s3)
    
    def checkSolution(self,rg,sg,rf,sf, show=True):
        """Проверка решения"""
        self.s.sendall((str(rg)+' '+str(sg)+' '+str(rf)+' '+str(sf)+'\n').encode())
        data=self.recv_until(b'\n')
        try:
            data=data.decode()
        except UnicodeDecodeError:
            print ('Error decoding unicode. Try connecting to server again.')
            return None
        if show:
            print (data)
        if data.find('flag')!=-1:
            return True
        else:
            data=self.recv_until(b'>')
            try:
                data=data.decode()
            except UnicodeDecodeError:
                print ('Error decoding unicode. Try connecting to server again.')
                return None
            if show:
                print (data)
            return False
    def __del__(self):
        self.s.close()

vs=VulnServerClient()
(p,q,g,y1,y2,m1,r1,s1,m2,r2,s2,m3,r3,s3)=vs.getChallenge()

Welcome to DSA signature forgery task
p=27359539911171676811372167899241610846069548287861652875241399472131280680871702241681042897128433137565026588370324497449395594240606889769623555526563155988070442998110299067825705185685160420133873717870489602025944410854313366123103351703194371186153876845288406143335283537035801804861759129506510371570198886385102938028499717788730023475065280871481593569246306328396315675619898339625134771385869793517372325476345748722649809545672927091947047246205383100530415365535589367604624247694006381901153567094193395122739304616761609503962727692355651888509858159396047919078799680509802732484956832732763295107767
g=4
y1=23737205675820451482672987113024766604345972027711620308489850279606386422607492700695859469934704071609890515011404587414797993301928633810972453313518588336801436612244154874402657430244077951719513475196416762424990793353670829997639876831747919251328804662822463926673948246821867843490121071024883283342057546461436322789600233743

1. Задание №1

Дано: 

+ ```(r1, s1)``` – подпись

+ ```p, q, g``` – параметры криптосистемы

+ Функция предсказания ```k```

+ ```Hash-функция``` для DSA

+ ```m1``` – сообщение

+ ```y1``` – публичный ключ

Из формулы подписи можем выразить искомый ```x:```

$x = \frac{s \cdot k - Hash(m1)}{r} \bmod q$


In [15]:
import hashlib

class predictable_DSA(DSA):
    def get_k(self):
        return int(time.time()) % (self.q-1) + 1
    
def H(m):
    if isinstance(m, str):
        m = m.encode()
    return bytes_to_long(hashlib.sha256(m).digest())

Функция атаки

In [16]:
def find_x1(r1, s1, p, q, hash_func, m1, y1):

    current_timestamp = int(time.time())

    for lag in range(-30, 30):
        candidate_timestamp = (current_timestamp + lag) % (q - 1) + 1

        try:
            inverted_r = pow(r1, -1, q)
            x_candidate = ((s1*candidate_timestamp - hash_func(m1)) * inverted_r) % q

            if pow(g, x_candidate, p) == y1:
                return x_candidate
            
        except ValueError:
            continue
    
    else:
        ValueError("No candidate found!")

x1 = find_x1(r1=r1, s1=s1, p=p, q=q, hash_func=H, m1=m1, y1=y1)
print(f"x1: {x1}")


x1: 12518722207370963036231843657163453231098501928332663079178034273686739069352168137865357221062128487144451850313083685210244404570905692515886663922631606283931359891926250176475129081668259267872697054501815660458546445692549332263850311851767683205781108910944496016892632127827652503960137389497155645090943173107083739377344791080935252843268722166194575418087623087632918025307128461186802548760585617522731620501912427049645494347823069793278033078091761259874158051045177691725259002812939053096837753396691605288748337796778208489605293088172589651169621956510122233936564176647763534815063519864436618249565


2. Задание 2

Дано:

+ Две подписи – ```(r2, s2)``` и ```(r3, s3)``` на одном закрытом ключе ```x2``` для сообщений ```m2``` и ```m3```

+ ```x2``` не меняется

+ ```p, q, g``` – параметры криптосистемы

+ Можем найти ```x2``` исходя из равенства при одном и том же ключе:

$$x=\frac{\mathit{Hash}(m_1)s_2-\mathit{Hash}(m_2)s_1}{r_2\cdot s_1-r_1\cdot s_2}\  mod \  q$$

In [17]:
def find_x2(p, q, g, y2, m2, r2, s2, m3, r3, s3, hash_func):

    assert r2 == r3

    h2 = hash_func(m2)
    h3 = hash_func(m3)

    numerator = (h2 * s3 - h3 * s2) % q
    denominator = (r2 * (s2 - s3)) % q

    inverted_denominator = pow(denominator, -1, q)
 
    x2 = (numerator * inverted_denominator) % q

    assert pow(g, x2, p) == y2

    return x2

x2 = find_x2(p=p, q=q, g=g, y2=y2, m2=m2, r2=r2, s2=s2, m3=m3, r3=r3, s3=s3, hash_func=H)
print(f"x2: {x2}")

x2: 9173258325467811488844481971052818369047415226293782382881551080517957709631919396796617839342256311041239853671229128026438367877676410685989955948842452727174741422969396754297840545299602846297872237289818532917161229336592288225301637145437029850759354747113300639413114825967817958832771631573130613141183173832505182891560367443815666874464075392730897458870607445033544579117278391240171324968093172200820941254328202427458906020246461607296184800017307930357643268612881034646338773444556609572422829135359399064812215483728628166018063468631336151124573861936921024302518731862641133502545513635401450563954


Создание подписей

In [19]:
def sign_message(m, x, p, q, g):
    h = H(m)
    while True:
        k = random.randint(1, q-1)
        r = pow(g, k, p) % q
        if r == 0:
            continue
        inv_k = pow(k, -1, q)
        s = ((h + x * r) * inv_k) % q
        if s != 0:
            break
    return (r, s)

vs = VulnServerClient()
(p, q, g, y1, y2, m1, r1, s1, m2, r2, s2, m3, r3, s3) = vs.getChallenge()

x1 = find_x1(r1=r1, s1=s1, p=p, q=q, hash_func=H, m1=m1, y1=y1)
print(f"x1: {x1}")

x2 = find_x2(p=p, q=q, g=g, y2=y2, m2=m2, r2=r2, s2=s2, m3=m3, r3=r3, s3=s3, hash_func=H)
print(f"x2: {x2}")

r_g, s_g = sign_message(b'give', x1, p, q, g)
r_f, s_f = sign_message(b'flag', x2, p, q, g)

Welcome to DSA signature forgery task
p=27359539911171676811372167899241610846069548287861652875241399472131280680871702241681042897128433137565026588370324497449395594240606889769623555526563155988070442998110299067825705185685160420133873717870489602025944410854313366123103351703194371186153876845288406143335283537035801804861759129506510371570198886385102938028499717788730023475065280871481593569246306328396315675619898339625134771385869793517372325476345748722649809545672927091947047246205383100530415365535589367604624247694006381901153567094193395122739304616761609503962727692355651888509858159396047919078799680509802732484956832732763295107767
g=4
y1=89292901854798604141303729603972198382626600054654811435691387309215203756338833710846524154037713468366797530936725065069097246234091045940272617229197282290708018122300124578858759934867897241412347075537547702211436870953086806491152120968136187005468347050397298617425704068016323432353216197994410429302409138781933114219841029729

In [20]:
vs.checkSolution(r_g,s_g,r_f,s_f)

Congratulations, your flag is: CRYPTOTRAINING{b3tt3r_m4k3_sur3_n0nc3_1s_s4f3}.



True